1. Setup

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
bool(os.getenv("GROQ_API_KEY"))


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model=os.getenv("MODEL_NAME", "openai/gpt-oss-120b"),
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.3,
)


In [ ]:
res = llm.invoke("hello")
res.content

2. State- Global State to pass the data b/a all the nodes

In [ ]:
from pydantic import BaseModel
class UserState(BaseModel):
    name: str = ""
    email: str = ""
    age: int = 0
state = UserState(name="Example User", email="learner@example.com", age=25)
state

2.1 Building the graph using langgraph

In [ ]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel

class MyState(BaseModel):
    message: str = ""
    is_passed: bool = False

def greet(state: MyState) -> dict[str, str | bool]:
    """Return a greeting and mark the state as passed."""
    return {
        "message": f"Good morning, {state.message}!",
        "is_passed": True,
    }

graph = StateGraph(MyState)
graph.add_node("Arshad", greet)

graph.add_edge(START, "Arshad")
graph.add_edge("Arshad", END)

final_graph = graph.compile()
final_graph.invoke({"message": "Hello"})


    

In [ ]:
final_graph

In [ ]:
res  = final_graph.invoke({"message": "Hello, Arshad!"})

In [ ]:
res

2.2 Multi Node Graph


In [ ]:
from random import randint

class ReviewState(BaseModel):
    text: str = ""
    score: int = 0
    comments: str = ""


# socre node

def score(state: ReviewState) -> dict[str, int]:
    return {"score": randint(1, 10)}

def add_comments(state: ReviewState) -> dict[str, str]:
    if state.score > 5:
        review_comment = "Good review"
    else:
        review_comment = "Bad review"
    return {"comments": review_comment}

#graph
graph  = StateGraph(ReviewState)
graph.add_node("score", score)
graph.add_node("comments", add_comments)

#Edges
graph.add_edge(START, "score")
graph.add_edge("score", "comments")
graph.add_edge("comments", END)

final_graph = graph.compile()

final_graph



In [ ]:
res = final_graph.invoke({})
res

3. 'add_message" reducers - for chat history

In [34]:
from typing import Annotated

from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

def custom_reducers(existing: list, new: list):
    all_msg = existing + new
    return all_msg[-1:]
class ChatState(BaseModel):
    messages: Annotated[list[AnyMessage], add_messages] = Field(default_factory=list)


def chat(state: ChatState) -> dict[str, list[AnyMessage]]:
    response = llm.invoke(state.messages)
    return {"messages": [response]}


graph = StateGraph(ChatState)
graph.add_node("chat", chat)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)

final_graph = graph.compile()

res = final_graph.invoke({"messages": [HumanMessage(content="Hi there!")]})


NameError: name 'llm' is not defined

4. Routing and condtional routing
